# AIC 2026 — Full embedding ablation on Kaggle

Notebook này chạy benchmark **embedding-only** trên 58 KIS chắc chắn:

- OpenCLIP, SigLIP2, Qwen3-VL-Embedding-2B
- full query và perspective n=3/5/7
- Top-100, exact `video_id + frame_id`
- xuất rank table, Recall@K, MRR và latency

Điều kiện bắt buộc: mỗi model đã có image-vector collection riêng trên Zilliz/Milvus.
Upload `dev_search_local.db` trong một Kaggle Dataset và attach Dataset đó vào notebook.
Không ghi secret trực tiếp vào notebook; dùng **Add-ons → Secrets**.

In [ ]:
# ========================= CONFIG DUY NHẤT =========================
REPO_URL = "https://github.com/zintomvn/Multimodal-Retrieval.git"
BRANCH = "experiment/embedding-ablation-molab"
REPO_DIR = "/kaggle/working/Multimodal-Retrieval"

# Kaggle Dataset đã attach. Để "" để notebook tự tìm trong /kaggle/input.
DATASET_INPUT_DIR = ""
DB_FILENAME = "dev_search_local.db"
GROUNDTRUTH_REL = "data/experiments/aic_2026_groundtruth/aic2026_all_confirmed.csv"

MODELS = ["openclip", "siglip2", "qwen3_vl"]
PERSPECTIVE_COUNTS = [3, 5, 7]
TOP_K = 100
SMOKE_QUERY_COUNT = 6
RUN_FULL_AFTER_SMOKE = True
FRAME_TOLERANCE = 0
MIN_EXPECTED_COLLECTION_ROWS = 300_000

# LLM is used only once to generate frozen perspectives.
PLANNER_PROFILE = "openai_gpt4o"
PLANNER_API_KEY_SECRET = "OPENAI_API_KEY"
# For Groq, use groq_gpt_oss_120b and GROQ_API_KEY instead.

# Chỉ OpenCLIP có server Python sẵn trong repo. SigLIP2/Qwen dùng URL từ Kaggle Secrets.
START_LOCAL_OPENCLIP = True
OPENCLIP_BASE_URL = "http://127.0.0.1:8001/v1"
BACKEND_BASE_URL = "http://127.0.0.1:8000"

MODEL_SPECS = {
    "openclip": {
        "model": "ViT-H-14-quickgelu-dfn5b",
        "dim": 1024,
        "collection": "keyframe_embeddings_clip_vith14_quickgelu_dfn5b_v2",
        "base_url_env": "CLIP_EMBEDDING_BASE_URL",
        "api_key_env": "",
    },
    "siglip2": {
        "model": "ViT-SO400M-16-SigLIP2-384-webli",
        "dim": 1152,
        "collection": "keyframe_embeddings_siglip2_so400m16_384_webli_openclip_1152_v1",
        "base_url_env": "SIGLIP2_EMBEDDING_BASE_URL",
        "api_key_env": "SIGLIP2_API_KEY",
    },
    "qwen3_vl": {
        "model": "Qwen/Qwen3-VL-Embedding-2B",
        "dim": 2048,
        "collection": "keyframe_embeddings_qwen3_vl_embedding_2b_2048_v1",
        "base_url_env": "QWEN3_VL_EMBEDDING_BASE_URL",
        "api_key_env": "QWEN3_VL_EMBEDDING_API_KEY",
    },
}

REQUIRED_SECRET_LABELS = [
    "MILVUS_URI",
    "MILVUS_TOKEN",
    PLANNER_API_KEY_SECRET,
    "SIGLIP2_EMBEDDING_BASE_URL",
    "SIGLIP2_API_KEY",
    "QWEN3_VL_EMBEDDING_BASE_URL",
    "QWEN3_VL_EMBEDDING_API_KEY",
]
# ==================================================================

In [ ]:
import os, shutil, subprocess, sys, time
from pathlib import Path

def run(command, *, cwd=None, env=None):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, cwd=cwd, env=env, check=True)

print("Python:", sys.version)
run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])

## 1. Clone code và cài dependency

In [ ]:
repo = Path(REPO_DIR)
if not repo.exists():
    run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(repo)])
else:
    run(["git", "fetch", "origin", BRANCH], cwd=repo)
    run(["git", "checkout", BRANCH], cwd=repo)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=repo)

run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "apps/backend/requirements.txt")])
if START_LOCAL_OPENCLIP and "openclip" in MODELS:
    run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "apps/backend/requirements-embedding-service.txt")])

## 2. Nạp Kaggle Secrets (không in giá trị)

In [ ]:
from kaggle_secrets import UserSecretsClient

secret_client = UserSecretsClient()
secret_status = {}
for label in REQUIRED_SECRET_LABELS:
    try:
        value = secret_client.get_secret(label)
    except Exception:
        value = ""
    if value:
        os.environ[label] = value
        secret_status[label] = "configured"
    else:
        secret_status[label] = "missing"

# OpenCLIP chạy nội bộ trong notebook.
os.environ["AGENT_LLM_PROFILE"] = PLANNER_PROFILE
if START_LOCAL_OPENCLIP:
    os.environ["CLIP_EMBEDDING_BASE_URL"] = OPENCLIP_BASE_URL

print(secret_status)

## 3. Tìm database và kiểm tra ground truth

In [ ]:
input_root = Path(DATASET_INPUT_DIR) if DATASET_INPUT_DIR else Path("/kaggle/input")
db_candidates = list(input_root.rglob(DB_FILENAME))
if not db_candidates:
    raise FileNotFoundError(
        f"Không tìm thấy {DB_FILENAME} trong {input_root}. "
        "Hãy attach Kaggle Dataset chứa file này."
    )

source_db = db_candidates[0]
target_db = repo / "data/dev_search_local.db"
target_db.parent.mkdir(parents=True, exist_ok=True)
if not target_db.exists() or target_db.stat().st_size != source_db.stat().st_size:
    shutil.copy2(source_db, target_db)

groundtruth = repo / GROUNDTRUTH_REL
assert groundtruth.exists(), f"Thiếu ground truth: {groundtruth}"
print("Database:", source_db, source_db.stat().st_size, "bytes")
print("Ground truth:", groundtruth)

run([
    sys.executable, str(repo / "scripts/run_embedding_ablation.py"),
    "--benchmark-csv", str(groundtruth), "--validate-only",
], cwd=repo)

## 4. Khởi động OpenCLIP text endpoint (nếu bật)

In [ ]:
openclip_process = None
if START_LOCAL_OPENCLIP and "openclip" in MODELS:
    openclip_log_path = Path("/kaggle/working/openclip.log")
    openclip_log = open(openclip_log_path, "w")
    openclip_process = subprocess.Popen(
        [
            sys.executable,
            str(repo / "apps/backend/scripts/serve_openclip_embeddings.py"),
            "--host", "127.0.0.1", "--port", "8001", "--device", "cuda",
        ],
        cwd=repo,
        stdout=openclip_log,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )
    print("OpenCLIP PID:", openclip_process.pid)
else:
    print("Không khởi động OpenCLIP local.")

## 5. Preflight embedding endpoints

In [ ]:
import httpx, math

def wait_embedding_endpoint(model_key, attempts=90):
    spec = MODEL_SPECS[model_key]
    base_url = os.getenv(spec["base_url_env"], "").rstrip("/")
    if not base_url:
        raise RuntimeError(f"Thiếu secret/env {spec['base_url_env']} cho {model_key}")
    api_key = os.getenv(spec["api_key_env"], "") if spec["api_key_env"] else ""
    headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}
    error = None
    for _ in range(attempts):
        try:
            with httpx.Client(timeout=120, headers=headers) as client:
                response = client.post(
                    f"{base_url}/embeddings",
                    json={"model": spec["model"], "input": ["a person cooking food"]},
                )
                response.raise_for_status()
                vector = response.json()["data"][0]["embedding"]
            if len(vector) != spec["dim"]:
                raise RuntimeError(f"{model_key}: expected dim={spec['dim']}, got {len(vector)}")
            if not vector or not all(math.isfinite(float(x)) for x in vector):
                raise RuntimeError(f"{model_key}: vector rỗng hoặc có NaN/Inf")
            print(model_key, "endpoint=OK", "dim=", len(vector))
            return
        except Exception as exc:
            error = exc
            time.sleep(2)
    raise RuntimeError(f"Endpoint {model_key} chưa sẵn sàng: {error}")

for model_key in MODELS:
    wait_embedding_endpoint(model_key)

## 6. Preflight Zilliz/Milvus collections

In [ ]:
from pymilvus import MilvusClient

milvus_uri = os.getenv("MILVUS_URI", "")
milvus_token = os.getenv("MILVUS_TOKEN", "")
if not milvus_uri or not milvus_token:
    raise RuntimeError("Thiếu MILVUS_URI hoặc MILVUS_TOKEN trong Kaggle Secrets")

milvus = MilvusClient(uri=milvus_uri, token=milvus_token)
collection_rows = {}
for model_key in MODELS:
    spec = MODEL_SPECS[model_key]
    name = spec["collection"]
    if not milvus.has_collection(collection_name=name):
        raise RuntimeError(f"Collection chưa tồn tại: {name}")
    description = milvus.describe_collection(collection_name=name)
    vector_field = next((f for f in description.get("fields", []) if f.get("name") == "vector"), {})
    params = vector_field.get("params") or {}
    dim = int(params.get("dim") or vector_field.get("dim") or 0)
    stats = milvus.get_collection_stats(collection_name=name)
    rows = int(stats.get("row_count") or 0)
    if dim != spec["dim"]:
        raise RuntimeError(f"{name}: expected dim={spec['dim']}, got {dim}")
    if rows < MIN_EXPECTED_COLLECTION_ROWS:
        raise RuntimeError(f"{name}: chỉ có {rows} vectors; cần ít nhất {MIN_EXPECTED_COLLECTION_ROWS}")
    collection_rows[model_key] = rows
    print(model_key, "collection=OK", "dim=", dim, "rows=", rows)

if len(set(collection_rows.values())) != 1:
    raise RuntimeError(f"Các model không có cùng số keyframe: {collection_rows}")

## 7. Khởi động backend local trong Kaggle

In [ ]:
backend_env = os.environ.copy()
backend_env.update({
    "DATABASE_URL": f"sqlite:///{target_db.resolve()}",
    "MODEL_REGISTRY_PATH": str(repo / "configs/model_registry.yaml"),
    "RETRIEVAL_PROFILES_PATH": str(repo / "configs/retrieval_profiles.yaml"),
    "AGENT_CONFIG_PATH": str(repo / "configs/agent.yaml"),
    "DATA_ROOT": str(repo / "data"),
    "SKIP_DB_INIT": "true",
    "MILVUS_CONNECT_TIMEOUT": "30",
    "MILVUS_SEARCH_TIMEOUT": "120",
})

backend_log_path = Path("/kaggle/working/backend.log")
backend_log = open(backend_log_path, "w")
backend_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd=repo / "apps/backend",
    stdout=backend_log,
    stderr=subprocess.STDOUT,
    env=backend_env,
)

for _ in range(60):
    try:
        response = httpx.get(f"{BACKEND_BASE_URL}/healthz", timeout=5)
        if response.status_code == 200:
            print("Backend=OK", response.json())
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError(backend_log_path.read_text(errors="replace")[-5000:])

## 8. Sinh và khóa 7 perspectives cho toàn bộ 58 KIS

In [ ]:
output_root = Path("/kaggle/working/embedding_ablation")
output_root.mkdir(parents=True, exist_ok=True)
perspectives_file = output_root / "perspectives.json"

run([
    sys.executable, str(repo / "scripts/run_embedding_ablation.py"),
    "--benchmark-csv", str(groundtruth),
    "--api-base", BACKEND_BASE_URL,
    "--perspective-counts", *map(str, PERSPECTIVE_COUNTS),
    "--perspectives-file", str(perspectives_file),
    "--output-dir", str(output_root / "planning"),
    "--plan-only",
], cwd=repo, env=backend_env)

## 9. Smoke test 6 query — bắt buộc phải pass

In [ ]:
smoke_dir = output_root / "smoke_6"
run([
    sys.executable, str(repo / "scripts/run_embedding_ablation.py"),
    "--benchmark-csv", str(groundtruth),
    "--api-base", BACKEND_BASE_URL,
    "--models", *MODELS,
    "--perspective-counts", *map(str, PERSPECTIVE_COUNTS),
    "--top-k", str(TOP_K),
    "--frame-tolerance", str(FRAME_TOLERANCE),
    "--limit", str(SMOKE_QUERY_COUNT),
    "--perspectives-file", str(perspectives_file),
    "--output-dir", str(smoke_dir),
    "--timeout-s", "300",
], cwd=repo, env=backend_env)

import pandas as pd
display(pd.read_csv(smoke_dir / "summary.csv"))
print((smoke_dir / "rank_table.md").read_text(encoding="utf-8"))

## 10. Full benchmark 58 KIS

In [ ]:
full_dir = output_root / "full_58"
if RUN_FULL_AFTER_SMOKE:
    run([
        sys.executable, str(repo / "scripts/run_embedding_ablation.py"),
        "--benchmark-csv", str(groundtruth),
        "--api-base", BACKEND_BASE_URL,
        "--models", *MODELS,
        "--perspective-counts", *map(str, PERSPECTIVE_COUNTS),
        "--top-k", str(TOP_K),
        "--frame-tolerance", str(FRAME_TOLERANCE),
        "--perspectives-file", str(perspectives_file),
        "--output-dir", str(full_dir),
        "--timeout-s", "300",
    ], cwd=repo, env=backend_env)
    display(pd.read_csv(full_dir / "summary.csv"))
else:
    print("RUN_FULL_AFTER_SMOKE=False — chỉ chạy smoke test.")

## 11. Đóng gói kết quả để tải về

In [ ]:
archive = shutil.make_archive(
    "/kaggle/working/embedding_ablation_results",
    "zip",
    output_root,
)
print("Download:", archive)
print("Smoke table:", smoke_dir / "rank_table.md")
if RUN_FULL_AFTER_SMOKE:
    print("Full metrics:", full_dir / "summary.csv")
    print("Full LaTeX table:", full_dir / "rank_table.tex")